In [ ]:
%run ./imports.py

In [ ]:
C_OPTICAL_FIBER_KM_PER_MS = (2/3) * (299792458 / 10**6)

## Load all RTTs

In [ ]:
ext_rtt_path    = "data/campus_trace_ext_rtt.csv"
flow_map_path   = "data/campus_trace_flow_map.csv"
geoloc_path     = "data/campus_trace_geolocation_map.csv"
puclients_ipinfo_path = "data/puclients_geolocation.csv"
ext_ip_map_path = "data/campus_trace_external_ip_map.csv"
int_ip_map_path = "data/campus_trace_internal_ip_map.csv"
conn_bytes_path = "data/conn_bytes_dict.pkl"

In [ ]:
ext_ip_map = {}
with open(ext_ip_map_path) as fp:
    for line in [l.strip() for l in fp.readlines()][1:]:
        tokens = line.split(",")
        ext_ip_map[tokens[0]] = tokens[1]
print(f"No. of external IPs: {len(ext_ip_map)}")

In [ ]:
int_ip_map = {}
with open(int_ip_map_path) as fp:
    for line in [l.strip() for l in fp.readlines()][1:]:
        tokens = line.split(",")
        int_ip_map[tokens[0]] = tokens[1]
print(f"No. of internal IPs: {len(int_ip_map)}")

In [ ]:
conn_bytes = None
with open("data/conn_bytes_dict.pkl", "rb") as fp:
    conn_bytes = pickle.load(fp)
    
def get_conn_bytes(row):
    conn_id = (row['Source_IP'], row['Source_Port'], row['Destination_IP'], row['Destination_Port'])
    return conn_bytes[conn_id]

In [ ]:
# Geolocation map
df_geo = pd.read_csv(geoloc_path, low_memory=False)

# Convert latitude and longitude to numeric, coercing errors to NaN
df_geo['Latitude'] = pd.to_numeric(df_geo['Latitude'], errors='coerce')
df_geo['Longitude'] = pd.to_numeric(df_geo['Longitude'], errors='coerce')

# Drop rows where latitude or longitude is NaN
df_geo = df_geo.dropna(subset=['Latitude', 'Longitude'])

df_geo = df_geo.rename(columns={'External_IP_ID': 'Destination_IP'})
print("df_geo row count:", df_geo.shape[0])
print(df_geo.head(n=1))

In [ ]:
# Geolocation map
df_geo_ipinfo = pd.read_csv(puclients_ipinfo_path, low_memory=False)

# Convert latitude and longitude to numeric, coercing errors to NaN
df_geo_ipinfo['Latitude'] = pd.to_numeric(df_geo_ipinfo['Latitude'], errors='coerce')
df_geo_ipinfo['Longitude'] = pd.to_numeric(df_geo_ipinfo['Longitude'], errors='coerce')

In [ ]:
amazon_va_prefixes = df_geo_ipinfo[
    (df_geo_ipinfo['Organization'].str.contains("Amazon.com", case=False, na=False))
    & (df_geo_ipinfo['Latitude'] == 39.0437)
    & (df_geo_ipinfo['Longitude'] == -77.4875)
]['Destination_Prefix'].unique().tolist()
print(f"{len(amazon_va_prefixes)} Amazon prefixes found")

In [ ]:
# Flows
## Note: Source IP/port is always the internal IP/port and destination IP/port is always the external IP/port
df_flows_without_geo = pd.read_csv(flow_map_path)
print("df_flows_without_geo row count:", df_flows_without_geo.shape[0])
print(df_flows_without_geo.head(n=1))

In [ ]:
df_flows = pd.merge(df_flows_without_geo, df_geo, on='Destination_IP', how='inner')
print(f"df_flows row count: {df_flows.shape[0]} ({round(df_flows.shape[0]*100/df_flows_without_geo.shape[0], 2)}%)")
print(df_flows.head(n=1))

In [ ]:
def internal_host_type(row):
    if row['Source_Port'] > 1023 and row['Destination_Port'] < 1024:
        return 'C' # Client in PU
    if row['Source_Port'] < 1024 and row['Destination_Port'] > 1023:
        return 'S' # Server in PU
    return 'N' # Neither

def get_trace_source_ip(row):
    if row['Source_IP'] in int_ip_map:
        return int_ip_map[row['Source_IP']]

def get_trace_destination_ip(row):
    if row['Destination_IP'] in ext_ip_map:
        return ext_ip_map[row['Destination_IP']]

In [ ]:
df_flows['Source_IP'] = df_flows.apply(get_trace_source_ip, axis=1)
df_flows['Destination_IP'] = df_flows.apply(get_trace_destination_ip, axis=1)
df_flows['Internal_Host_Type'] = df_flows.apply(internal_host_type, axis=1)
df_flows['Connection_Bytes'] = df_flows.apply(get_conn_bytes, axis=1)
df_flows['Destination_Prefix'] = df_flows['Destination_IP'].apply(lambda x: ".".join(x.split(".")[:-1]+["0"]))
print(f"df_flows row count: {df_flows.shape[0]}")
print(df_flows.head(n=1))

In [ ]:
columns_tocorrect = ['Continent', 'Country', 'Latitude', 'Longitude']
merged_df = df_flows.merge(df_geo_ipinfo, on='Destination_Prefix', how='left', suffixes=('', '_new'))
for col in columns_tocorrect:
    merged_df[col] = merged_df[col + '_new'].combine_first(merged_df[col])
df_flows = merged_df[df_flows.columns]
merged_df = None
print(f"df_flows row count: {df_flows.shape[0]}")
print(df_flows.head(n=1))

In [ ]:
# RTTs
df_rtts = pd.read_csv(ext_rtt_path)
print("df_rtts row count:", df_rtts.shape[0])
print(df_rtts.head(n=1))

In [ ]:
df_rtts_minrtt = df_rtts.groupby('Flow_ID').agg(
    minRTT_ms=('RTT_ms', 'min'),
    RTT_Count=('RTT_ms', 'count')
).reset_index()
print("df_rtts_minrtt row count:", df_rtts_minrtt.shape[0])
print(df_rtts_minrtt.head(n=1))

In [ ]:
df_rtts_minrtt_filtered = df_rtts_minrtt[ df_rtts_minrtt['RTT_Count'] >= 10 ]
print(f"df_rtts_minrtt_filtered row count: {df_rtts_minrtt_filtered.shape[0]}"
      + f" ({round(df_rtts_minrtt_filtered.shape[0]*100/df_rtts_minrtt.shape[0], 2)}%)")
print(df_rtts_minrtt_filtered.head(n=1))

In [ ]:
df_flows_minrtt = pd.merge(df_flows, df_rtts_minrtt_filtered, on='Flow_ID', how='inner')
print("df_flows_minrtt row count:", df_flows_minrtt.shape[0])
print(df_flows_minrtt.head(n=1))

In [ ]:
df_prefixes_minrtt_nonagg = df_flows_minrtt.groupby('Destination_Prefix').agg({
    'Connection_Bytes': list,
    'Internal_Host_Type': list,
    'Flow_ID': list,
    'Source_IP': list,
    'Destination_IP': list,
    'Source_Port': list,
    'Destination_Port': list,
    'Continent': list,
    'Country': list,
    'Latitude': list,
    'Longitude': list,
    'minRTT_ms': list,
    'RTT_Count': list
}).reset_index()
print(f"Shape of df_prefixes_minrtt_nonagg: {df_prefixes_minrtt_nonagg.shape[0]}")
print(df_prefixes_minrtt_nonagg.head(n=1))

In [ ]:
def identity(x):
    return x

def add_elements(x):
    return sum(x)

def unique_elements(x):
    return list(set(x))

def count_elements(x):
    return len(x)

def count_unique_elements(x):
    return len(list(set(x)))

def min_elements(x):
    return min(x)

In [ ]:
# Define aggregations for each column
aggregations = {
    'Destination_Prefix': identity,
    'Connection_Bytes': add_elements,
    'Internal_Host_Type': unique_elements,
    'Flow_ID': count_unique_elements,
    'Source_IP': count_unique_elements,
    'Destination_IP': count_unique_elements,
    'Source_Port': count_unique_elements,
    'Destination_Port': count_unique_elements,
    'Continent': unique_elements,
    'Country': unique_elements,
    'Latitude': unique_elements,
    'Longitude': unique_elements,
    'minRTT_ms': min_elements,
    'RTT_Count': add_elements
}

df_prefixes_minrtt_agg = pd.DataFrame({
    col: df_prefixes_minrtt_nonagg[col].apply(aggregations[col]) for col in list(aggregations.keys())
})

df_prefixes_minrtt_agg = df_prefixes_minrtt_agg.rename(columns={
    'Connection_Bytes': 'Bytes',
    'Internal_Host_Type': 'Internal_Host_Type_Unique',
    'Flow_ID': 'Flow_Count',
    'Source_IP': 'Source_IP_Unique_Count',
    'Destination_IP': 'Destination_IP_Unique_Count',
    'Source_Port': 'Source_Port_Unique_Count',
    'Destination_Port': 'Destination_Port_Unique_Count',
    'Continent': 'Continents_Unique',
    'Country': 'Country_Unique',
    'Latitude': 'Latitude_Unique',
    'Longitude': 'Longitude_Unique'
})

print(f"Shape of df_prefixes_minrtt_agg: {df_prefixes_minrtt_agg.shape[0]}"
     + f" ({round(df_prefixes_minrtt_agg.shape[0]*100/df_prefixes_minrtt_nonagg.shape[0], 2)}%)")
print(df_prefixes_minrtt_agg.head(n=1))

In [ ]:
df_prefixes_minrtt_agg_filtered = df_prefixes_minrtt_agg[
    (df_prefixes_minrtt_agg['Latitude_Unique'].apply(lambda x: len(x) == 1))
    & (df_prefixes_minrtt_agg['Longitude_Unique'].apply(lambda x: len(x) == 1))
]
print(f"Shape of df_prefixes_minrtt_agg_filtered: {df_prefixes_minrtt_agg_filtered.shape[0]}" \
      + f" ({round(df_prefixes_minrtt_agg_filtered.shape[0]*100/df_prefixes_minrtt_agg.shape[0], 2)}%)")
print(df_prefixes_minrtt_agg_filtered.head(n=1))

In [ ]:
def get_distance_from_princeton(row):
    src_coord = (40.343899, -74.660049)
    dst_coord = (float(row['Latitude']), float(row['Longitude']))
    distance = geodesic(src_coord, dst_coord).km
    return distance

In [ ]:
def compute_rtt_lb(row):
    dist_rt_m = 2 * row['Geodesic_Distance_km'] * 1000
    c = 299792458
    rtt = dist_rt_m/c
    return rtt * 1000

In [ ]:
df_prefixes_minrtt = df_prefixes_minrtt_agg_filtered.copy()
df_prefixes_minrtt['Continent'] = df_prefixes_minrtt['Continents_Unique'].apply(lambda x: x[0])
df_prefixes_minrtt['Country'] = df_prefixes_minrtt['Country_Unique'].apply(lambda x: x[0])
df_prefixes_minrtt['Latitude'] = df_prefixes_minrtt['Latitude_Unique'].apply(lambda x: x[0])
df_prefixes_minrtt['Longitude'] = df_prefixes_minrtt['Longitude_Unique'].apply(lambda x: x[0])
df_prefixes_minrtt['Geodesic_Distance_km'] = df_prefixes_minrtt.apply(get_distance_from_princeton, axis=1)

df_prefixes_minrtt = df_prefixes_minrtt[[
    'Destination_Prefix', 'Bytes', 'Internal_Host_Type_Unique',
    'Flow_Count', 'Continent', 'Country', 'Latitude', 'Longitude', 'Geodesic_Distance_km', 'RTT_Count', 'minRTT_ms'
]]

df_prefixes_minrtt['minRTT_Lower_Bound_ms'] = df_prefixes_minrtt.apply(compute_rtt_lb, axis=1)

print(f"Shape of df_prefixes_minrtt: {df_prefixes_minrtt.shape[0]}")
print(df_prefixes_minrtt.head(n=1))

## Client in PU

In [ ]:
df_prefixes_minrtt_puclients = df_prefixes_minrtt[
    df_prefixes_minrtt['Internal_Host_Type_Unique'].apply(
        lambda x: len(x) == 1 and x[0] == 'C')]
print(f"Shape of df_prefixes_minrtt_puclients: {df_prefixes_minrtt_puclients.shape[0]}")
print(df_prefixes_minrtt_puclients.head(n=1))

In [ ]:
rtt_counts = df_prefixes_minrtt_puclients['RTT_Count'].tolist()
pu.cdf(rtt_counts, xlabel="Sample Count", title="Dist. of RTT sample counts", xscale="log")

In [ ]:
# Filter
df_prefixes_minrtt_puclients_filtered = df_prefixes_minrtt_puclients[
    (df_prefixes_minrtt_puclients['Flow_Count'] >= 2)
    & (df_prefixes_minrtt_puclients['minRTT_ms'] >= df_prefixes_minrtt_puclients['minRTT_Lower_Bound_ms'])
    & (df_prefixes_minrtt_puclients['minRTT_ms'] <= 1337)
    & ~(df_prefixes_minrtt_puclients['Destination_Prefix'].isin(amazon_va_prefixes))
]
print(f"Shape of df_prefixes_minrtt_puclients_filtered: {df_prefixes_minrtt_puclients_filtered.shape[0]}" \
      + f" ({round(df_prefixes_minrtt_puclients_filtered.shape[0]*100/df_prefixes_minrtt_puclients.shape[0], 2)}%)")
print(df_prefixes_minrtt_puclients_filtered.head(n=1))

In [ ]:
distances = df_prefixes_minrtt_puclients_filtered['Geodesic_Distance_km'].tolist()
minrtts = df_prefixes_minrtt_puclients_filtered['minRTT_ms'].tolist()
distances_sorted, minrtts_sorted = zip(*sorted(zip(distances, minrtts)))
distances_sorted = list(distances_sorted)
minrtts_sorted = list(minrtts_sorted)

In [ ]:
pu.cdf(distances, title="Dist. of distance", xlabel="Distance from Princeton (km)")

In [ ]:
pu.cdf(minrtts, title="Dist. of minRTTs", xlabel="minRTT (ms)")

In [ ]:
def compute_rtts_at_ratio(ds, r):
    rtts = []
    for d in ds:
        dist_rt_m = 2 * d * 1000
        c = 299792458
        rtt_c = dist_rt_m * 1000 / (2*c/3)
        rtt = rtt_c * r
        rtts.append(rtt)
    return rtts

In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(distances_sorted, minrtts_sorted, color="blue", marker="o", s=15)
xs = []
ys = []
curvelabels = []
for r in [1, 1.5, 4, 10]:
    xs.append(distances_sorted)
    ys.append(compute_rtts_at_ratio(distances_sorted, r))
    if r == 1:
        curvelabels.append("rtt_lb")
    else:
        curvelabels.append(f"{r}rtt_lb")
loc = "upper left"
xlabel = "Distance (km)"
ylabel = "minRTT (ms)"
title = "Distance vs. minRTT"
# ylim = (0, 1338)
for i, (x, y) in enumerate(zip(xs, ys)):
    plt.plot(x, y, linewidth=5, linestyle='--', label=curvelabels[i])
# plt.ylim(ylim[0], ylim[1])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.legend(loc=loc, ncol=3)
plt.title(title)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import linregress

y = np.array([2*d*1000 for d in distances_sorted])
x = np.array([t/1000 for t in minrtts_sorted])

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(x, y)

print(f"Slope: {slope}, Intercept: {intercept}, R-squared: {r_value**2}")
print(f"Ratio of c: {slope/299792458}")

# Generate fitted line
y_fit = slope * x + intercept

# Plot the data and fitted line
plt.scatter(x, y, label="Data Points")
plt.plot(x, y_fit, color='red', label="Fitted Line")
plt.legend()
plt.show()

In [ ]:
def compute_ub_outlier_pct(ds, mrs, r):
    outlier = 0
    for dist, rtt in zip(ds, mrs):
        dist_rt_m = 2 * dist * 1000
        c = 299792458
        rtt_c = dist_rt_m * 1000 / (2*c/3)
        rtt_ub = rtt_c * r
        if rtt > rtt_ub:
            if dist > 0:
                outlier += 1
    return outlier * 100 / len(mrs)

In [ ]:
def compute_lb_outlier_pct(ds, mrs):
    outlier = 0
    for dist, rtt in zip(ds, mrs):
        dist_rt_m = 2 * dist * 1000
        c = 299792458
        rtt_c = dist_rt_m * 1000 / c
        rtt_lb = 1.5 * rtt_c
        if rtt < rtt_lb:
            outlier += 1
    return outlier * 100 / len(mrs)

In [ ]:
plt.figure(figsize=(9,6))
x = []
y = []
for r in np.arange(3, 15, 0.5):
    x.append(r)
    y.append(compute_ub_outlier_pct(distances_sorted, minrtts_sorted, r))
loc = "upper left"
xlabel = "r where RTT_ub = RTT_lb x r"
ylabel = "Outliers (%)"
title = "Upper bound speed vs. outliers"
plt.plot(x, y, linewidth=5)
# plt.ylim(ylim[0], ylim[1])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.tight_layout()
plt.show()

In [ ]:
print(f"Outliers below the lower bound: {round(compute_lb_outlier_pct(distances_sorted, minrtts_sorted), 2)}%")

## Server in PU

In [ ]:
df_prefixes_minrtt_puservers = df_prefixes_minrtt[
    df_prefixes_minrtt['Internal_Host_Type_Unique'].apply(
        lambda x: len(x) == 1 and x[0] == 'S')]

In [ ]:
rtt_counts = df_prefixes_minrtt_puservers['RTT_Count'].tolist()
pu.cdf(rtt_counts, xlabel="Sample Count", title="Dist. of RTT sample counts", xscale="log")

In [ ]:
# Filter
df_prefixes_minrtt_puservers_filtered = df_prefixes_minrtt_puservers[
    (df_prefixes_minrtt_puservers['RTT_Count'] >= 10)
    & (df_prefixes_minrtt_puservers['minRTT_ms'] >= df_prefixes_minrtt_puservers['minRTT_Lower_Bound_ms'])
    & (df_prefixes_minrtt_puservers['minRTT_ms'] <= 1337)
]

print(f"Shape of df_prefixes_minrtt_puservers_filtered: {df_prefixes_minrtt_puservers_filtered.shape[0]}" \
      + f" ({round(df_prefixes_minrtt_puservers_filtered.shape[0]*100/df_prefixes_minrtt_puservers.shape[0], 2)}%)")
print(df_prefixes_minrtt_puservers_filtered.head(n=1))

In [ ]:
distances = df_prefixes_minrtt_puservers_filtered['Geodesic_Distance_km'].tolist()
minrtts = df_prefixes_minrtt_puservers_filtered['minRTT_ms'].tolist()
distances_sorted, minrtts_sorted = zip(*sorted(zip(distances, minrtts)))
distances_sorted = list(distances_sorted)
minrtts_sorted = list(minrtts_sorted)

In [ ]:
rtts = {}
for r in range(1, 21):
    rtts[r] = compute_rtts_at_ratio(distances_sorted, r)

In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(distances_sorted, minrtts_sorted, color="blue", marker="o", s=15)
xs = []
ys = []
curvelabels = []
for r in range(1, 6, 1):
    xs.append(distances_sorted)
    ys.append(rtts[r])
    curvelabels.append(f"r={r}")
loc = "upper left"
xlabel = "Distance (km)"
ylabel = "minRTT (ms)"
title = "Server in PU: Distance vs. minRTT"
# ylim = (0, 1338)
for i, (x, y) in enumerate(zip(xs, ys)):
    plt.plot(x, y, linewidth=5, label=curvelabels[i])
# plt.ylim(ylim[0], ylim[1])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.legend(loc=loc, ncol=3)
plt.title(title)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import linregress

y = np.array([2*d*1000 for d in distances_sorted])
x = np.array([t/1000 for t in minrtts_sorted])

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(x, y)

print(f"Slope: {slope}, Intercept: {intercept}, R-squared: {r_value**2}")
print(f"Ratio of c: {slope/299792458}")

# Generate fitted line
y_fit = slope * x + intercept

# Plot the data and fitted line
plt.scatter(x, y, label="Data Points")
plt.plot(x, y_fit, color='red', label="Fitted Line")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9,6))
x = []
y = []
for r in np.arange(5, 15, 0.1):
    x.append(r)
    y.append(compute_ub_outlier_pct(distances_sorted, minrtts_sorted, r))
loc = "upper left"
xlabel = "r where speed = c/r"
ylabel = "Outliers (%)"
title = "Upper bound speed vs. outliers"
plt.plot(x, y, linewidth=5)
# plt.ylim(ylim[0], ylim[1])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.tight_layout()
plt.show()

## All types of prefixes

In [ ]:
rtt_counts = df_prefixes_minrtt['RTT_count'].tolist()
pu.cdf(rtt_counts, xlabel="Sample Count", title="Dist. of RTT sample counts", xscale="log")

In [ ]:
# Filter
df_prefixes_minrtt_filtered = df_prefixes_minrtt[
    (df_prefixes_minrtt['RTT_count'] >= 10)
    & (df_prefixes_minrtt['minRTT_ms'] >= df_prefixes_minrtt['minRTT_Lower_Bound_ms'])
    & (df_prefixes_minrtt['minRTT_ms'] <= 1337)
]

print(f"Shape of df_prefixes_minrtt_filtered: {df_prefixes_minrtt_filtered.shape[0]}" \
      + f" ({round(df_prefixes_minrtt_filtered.shape[0]*100/df_prefixes_minrtt.shape[0], 2)}%)")
print(df_prefixes_minrtt_filtered.head(n=1))

In [ ]:
distances = df_prefixes_minrtt_filtered['Geodesic_Distance_km'].tolist()
minrtts = df_prefixes_minrtt_filtered['minRTT_ms'].tolist()
distances_sorted, minrtts_sorted = zip(*sorted(zip(distances, minrtts)))
distances_sorted = list(distances_sorted)
minrtts_sorted = list(minrtts_sorted)

In [ ]:
rtts = {}
for r in range(1, 21):
    rtts[r] = compute_rtt_at_ratio_of_c(distances_sorted, r)

In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(distances_sorted, minrtts_sorted, color="blue", marker="o", s=15)
xs = []
ys = []
curvelabels = []
for r in range(1, 21, 2):
    xs.append(distances_sorted)
    ys.append(rtts[r])
    curvelabels.append(f"c/{r}")
loc = "upper left"
xlabel = "Distance (km)"
ylabel = "minRTT (ms)"
title = "Distance vs. minRTT"
# ylim = (0, 1338)
for i, (x, y) in enumerate(zip(xs, ys)):
    plt.plot(x, y, linewidth=5, label=curvelabels[i])
# plt.ylim(ylim[0], ylim[1])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.legend(loc=loc, ncol=3)
plt.title(title)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import linregress

y = np.array([2*d*1000 for d in distances_sorted])
x = np.array([t/1000 for t in minrtts_sorted])

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(x, y)

print(f"Slope: {slope}, Intercept: {intercept}, R-squared: {r_value**2}")
print(f"Ratio of c: {slope/299792458}")

# Generate fitted line
y_fit = slope * x + intercept

# Plot the data and fitted line
plt.scatter(x, y, label="Data Points")
plt.plot(x, y_fit, color='red', label="Fitted Line")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9,6))
x = []
y = []
for r in np.arange(9, 45, 0.1):
    x.append(r)
    y.append(compute_ub_outlier_pct(distances_sorted, minrtts_sorted, r))
loc = "upper left"
xlabel = "r where speed = c/r"
ylabel = "Outliers (%)"
title = "Upper bound speed vs. outliers"
plt.plot(x, y, linewidth=5)
# plt.ylim(ylim[0], ylim[1])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.tight_layout()
plt.show()

## MLab

In [ ]:
mlab_path = "data/ndt7_20241201_20241210_median.csv"

In [ ]:
def get_distance_between_coordinates(row):
    src_coord = (float(row['slat']), float(row['slon']))
    dst_coord = (float(row['clat']), float(row['clon']))
    try:
        distance = geodesic(src_coord, dst_coord).km
    except Exception as e:
        distance = -1
    return distance

In [ ]:
df_mlab = pd.read_csv(mlab_path, low_memory=False)
df_mlab['distance_km'] = df_mlab.apply(get_distance_between_coordinates, axis=1)
# df_mlab = df_mlab[(df_mlab['distance_km'] > 0.0) & (df_mlab['minrtt'] <= 1338)]
df_mlab = df_mlab[df_mlab['distance_km'] > 0.0]
print(df_mlab.shape[0])
df_mlab.head(n=5)

In [ ]:
distances = df_mlab['distance_km'].tolist()
minrtts = df_mlab['minrtt'].tolist()
distances_sorted, minrtts_sorted = zip(*sorted(zip(distances, minrtts)))
distances_sorted = list(distances_sorted)
minrtts_sorted = list(minrtts_sorted)


In [ ]:
pu.cdf(minrtts_sorted, {"title": "Dist. of MLab MinRTTs", "xlabel": "MinRTT (ms)"})

In [ ]:
rtts = {}
for r in range(1, 21):
    rtts[r] = compute_rtts_at_ratio(distances_sorted, r)

In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(distances_sorted, minrtts_sorted, color="blue", marker="o", s=15)
xs = []
ys = []
curvelabels = []
for r in range(1, 6, 1):
    xs.append(distances_sorted)
    ys.append(rtts[r])
    curvelabels.append(f"r={r}")
loc = "upper left"
xlabel = "Distance (km)"
ylabel = "minRTT (ms)"
title = "MLab NDT7 Data: Distance vs. minRTT"
# ylim = (0, 1338)
for i, (x, y) in enumerate(zip(xs, ys)):
    plt.plot(x, y, linewidth=5, label=curvelabels[i])
# plt.ylim(ylim[0], ylim[1])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.legend(loc=loc, ncol=3)
plt.title(title)
plt.tight_layout()
plt.show()

In [ ]:
_, ax, props = pu.setup_plot({"figsize": (11, 6), "xlabel": "Geodesic Distance (km)", "ylabel": "Minimum\nOne-Way Delay (ms)", "ylim": (-10, 260)})
ax.scatter(distances_sorted,
           [rtt/2 for rtt in minrtts_sorted],
           marker="x", color="blue", label="MLab NDT7 Measurements", s=props["markersize"])
# ax.plot(smooth_distances, predicted_owds, color="darkorange", label="Fitted Regression Line", linewidth=props["linewidth"])
# ax.plot(smooth_distances, theoretical_minowds, color="darkgrey", label="Speed-of-Light", linewidth=props["linewidth"])
ax.legend()
ax = pu.add_properties(ax, props)
pu.close_plot(ax, props)

In [ ]:
from scipy.stats import linregress

y = np.array([2*d*1000 for d in distances_sorted])
x = np.array([t/1000 for t in minrtts_sorted])

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(x, y)

print(f"Slope: {slope}, Intercept: {intercept}, R-squared: {r_value**2}")
print(f"Ratio of c: {slope/299792458}")

# Generate fitted line
y_fit = slope * x + intercept

# Plot the data and fitted line
plt.scatter(x, y, label="Data Points")
plt.plot(x, y_fit, color='red', label="Fitted Line")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9,6))
x = []
y = []
for r in np.arange(15, 45, 0.1):
    x.append(r)
    y.append(compute_ub_outlier_pct(distances_sorted, minrtts_sorted, r))
loc = "upper left"
xlabel = "r where speed = c/r"
ylabel = "Outliers (%)"
title = "Upper bound speed vs. outliers"
plt.plot(x, y, linewidth=5)
# plt.ylim(ylim[0], ylim[1])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.tight_layout()
plt.show()

## WIDE dataset

In [ ]:
df_wide = pd.read_csv("datasets/wide/wide_rtts.csv")
df_wide.head(n=1)

In [ ]:
print(df_wide["tcp.stream"].value_counts())

In [ ]:
df_wide[df_wide["tcp.stream"] == 375].head()

In [ ]:
df_wide[df_wide["tcp.stream"] == 100]["tcp.analysis.ack_rtt"].tolist()

In [ ]:
geodesic((41.8500,-87.6500), (41.8500,-87.6500)).km * 2 / C_OPTICAL_FIBER_KM_PER_MS

### Check OIT prefix relationships

In [ ]:
import ipaddress

In [ ]:
def list_all_subnets(supernet, mask):
    # Define the /21 prefix
    prefix = ipaddress.IPv4Network(supernet)
    # List all /24 prefixes within the /21 prefix
    subnets = list(prefix.subnets(new_prefix=mask))
    # Print the results
    for subnet in subnets:
        print(subnet)

In [ ]:
def is_subnet_of(supernet, subnet):
    return ipaddress.IPv4Network(supernet).supernet_of(ipaddress.IPv4Network(subnet))

In [ ]:
def binary(n):
    print(bin(n)[2:].zfill(8))

In [ ]:
list_all_subnets("140.180.232.0/21", 24)

In [ ]:
print(f"Is {supernet} a subnet of {subnet}?: " + str(is_subnet_of("140.180.0.0/16", "140.180.232.0/21")))

In [ ]:
binary(6)
binary(8)
binary(16)
binary(28)
binary(48)
binary(50)

In [ ]:
list_all_subnets("10.0.0.0/10", 15)